# Multiple data sets
The same sample is often measured more than once, for example with two incident energies on the same instrument. The data sets then have different resolutions and energy windows, and some features of the sample are only visible in one of them.

Here we fit two such data sets together. The sample has three Lorentzians: `L1` is narrow and only resolved in the fine-resolution data; `L2` is seen in both; `L3` is broad and only fits inside the energy window of the coarse-resolution data. We share `L2` between the two fits, so its parameters are determined by both data sets at once.

In [1]:
# Imports
import numpy as np
import scipp as sc

import easydynamics as edyn

# Make the plots interactive
%matplotlib widget

We generate two artificial data sets of the same sample. The fine setting has a resolution width of 0.01 meV and an energy window of ±1 meV; the coarse setting has 0.1 meV and ±8 meV. The two `Q` grids overlap, but each has one `Q` value the other does not. Real data would be loaded into an `Experiment` as in the other tutorials.

A Lorentzian convolved with a Gaussian resolution is a `Voigt`, so we can write down the measured line shapes directly. The width of `L2` grows as $Q^2$, like Brownian diffusion.

In [2]:
def make_experiment(
    name: str, Q: sc.Variable, energy: sc.Variable, resolution_width: float
) -> edyn.Experiment:
    intensity = np.zeros((Q.size, energy.size))
    for i, q in enumerate(Q.values):
        components = edyn.ComponentCollection(
            components=[
                edyn.Voigt(
                    name='L1', area=0.3, lorentzian_width=0.01, gaussian_width=resolution_width
                ),
                edyn.Voigt(
                    name='L2',
                    area=1.0,
                    lorentzian_width=0.08 * q**2,
                    gaussian_width=resolution_width,
                ),
                edyn.Voigt(
                    name='L3', area=2.0, lorentzian_width=4.0, gaussian_width=resolution_width
                ),
                edyn.Polynomial(coefficients=[0.02]),
            ]
        )
        intensity[i] = components.evaluate(energy.values)

    sigma = 0.01 * intensity.max()
    noise = np.random.default_rng(1).normal(0.0, sigma, intensity.shape)
    data = sc.DataArray(
        data=sc.array(
            dims=['Q', 'energy'],
            values=intensity + noise,
            variances=np.full(intensity.shape, sigma**2),
        ),
        coords={'Q': Q, 'energy': energy},
    )
    return edyn.Experiment(display_name=name, data=data)


fine_experiment = make_experiment(
    'Fine resolution',
    Q=sc.linspace('Q', 0.8, 2.2, 8, unit='1/angstrom'),
    energy=sc.linspace('energy', -1.0, 1.0, 201, unit='meV'),
    resolution_width=0.01,
)
coarse_experiment = make_experiment(
    'Coarse resolution',
    Q=sc.linspace('Q', 1.0, 2.4, 8, unit='1/angstrom'),
    energy=sc.linspace('energy', -8.0, 8.0, 161, unit='meV'),
    resolution_width=0.1,
)

In [3]:
fine_experiment.plot_data(slicer=True)

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [4]:
coarse_experiment.plot_data(slicer=True)

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

We create the three Lorentzians once and build a `SampleModel` for each data set from them. Appending the same `L2` object to both models is what marks it as shared. In the coarse data `L1` is much narrower than the resolution, so there it is indistinguishable from an elastic line and we model it with a `DeltaFunction`. `L3` is so broad that in the fine data it is a flat background, which the `BackgroundModel` absorbs.

In [5]:
L1 = edyn.Lorentzian(name='L1', area=0.3, width=0.01)
L2 = edyn.Lorentzian(name='L2', area=1.0, width=0.1)
L3 = edyn.Lorentzian(name='L3', area=2.0, width=3.0)
elastic = edyn.DeltaFunction(name='Elastic', area=0.3)

fine_model = edyn.SampleModel(components=edyn.ComponentCollection(components=[L1, L2]))
coarse_model = edyn.SampleModel(components=edyn.ComponentCollection(components=[elastic, L2, L3]))

Each data set has its own resolution and background, so each gets its own `InstrumentModel`. We take the resolutions as known and fix them.

In [6]:
def make_instrument_model(resolution_width: float) -> edyn.InstrumentModel:
    resolution_model = edyn.ResolutionModel(components=edyn.Gaussian(width=resolution_width))
    resolution_model.fix_all_parameters()
    background_model = edyn.BackgroundModel(components=edyn.Polynomial(coefficients=[0.02]))
    return edyn.InstrumentModel(
        resolution_model=resolution_model, background_model=background_model
    )


fine_analysis = edyn.Analysis(
    display_name='Fine resolution',
    experiment=fine_experiment,
    sample_model=fine_model,
    instrument_model=make_instrument_model(0.01),
)
coarse_analysis = edyn.Analysis(
    display_name='Coarse resolution',
    experiment=coarse_experiment,
    sample_model=coarse_model,
    instrument_model=make_instrument_model(0.1),
)

As always, we check the start guesses before fitting.

In [7]:
fine_analysis.plot_data_and_model()

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [8]:
coarse_analysis.plot_data_and_model()

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

A `JointAnalysis` fits several analyses together. At every `Q` value the two analyses have in common, the `L2` parameters of the coarse analysis follow those of the fine analysis, so they are fitted once against both data sets. At the one `Q` value only the coarse data has, the coarse analysis keeps its own copy of `L2`. Components that appear in only one model are fitted as usual.

In [9]:
joint_analysis = edyn.JointAnalysis(analyses=[fine_analysis, coarse_analysis])
print(joint_analysis.shared_components)

[Lorentzian(name = L2, display_name = L2, x_unit = meV, y_unit = dimensionless,
    area = <Parameter 'L2 area': 1.0000 meV, bounds=[0.0:inf]>,
    center = <Parameter 'L2 center': 0.0000 meV (fixed), bounds=[-inf:inf]>,
    width = <Parameter 'L2 width': 0.1000 meV, bounds=[1e-10:inf]>)]


`fit` fits all `Q` of both data sets simultaneously.

In [10]:
joint_analysis.fit()
fine_analysis.plot_data_and_model()

/home/runner/work/dynamics-lib/dynamics-lib/src/easydynamics/sample_model/component_collection.py:393: UserWarning: The Polynomial with unique_name Polynomial_28 has negative values, which may not be physically meaningful.
  gen = (component.evaluate(x, output=output) for component in self)


/home/runner/work/dynamics-lib/dynamics-lib/src/easydynamics/sample_model/component_collection.py:393: UserWarning: The Polynomial with unique_name Polynomial_30 has negative values, which may not be physically meaningful.
  gen = (component.evaluate(x, output=output) for component in self)


WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [11]:
coarse_analysis.plot_data_and_model()

/home/runner/work/dynamics-lib/dynamics-lib/src/easydynamics/sample_model/component_collection.py:393: UserWarning: The Polynomial with unique_name Polynomial_28 has negative values, which may not be physically meaningful.
  gen = (component.evaluate(x, output=output) for component in self)
/home/runner/work/dynamics-lib/dynamics-lib/src/easydynamics/sample_model/component_collection.py:393: UserWarning: The Polynomial with unique_name Polynomial_30 has negative values, which may not be physically meaningful.
  gen = (component.evaluate(x, output=output) for component in self)
/home/runner/work/dynamics-lib/dynamics-lib/src/easydynamics/analysis/analysis1d.py:835: UserWarning: The Polynomial with unique_name Polynomial_28 has negative values, which may not be physically meaningful.
  return components.evaluate(energy_with_offset)
/home/runner/work/dynamics-lib/dynamics-lib/src/easydynamics/analysis/analysis1d.py:835: UserWarning: The Polynomial with unique_name Polynomial_30 has negati

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

Where the `Q` values agree, the shared parameters belong to the first analysis in the list and are reported there. `parameters_to_dataset` on the `JointAnalysis` collects the parameters of the shared components over the `Q` values of all the data sets: eight from each, of which seven are in common.

In [12]:
joint_analysis.parameters_to_dataset()

<scipp.Dataset>
Dimensions: Sizes[Q:9, ]
Coordinates:
* Q                         float64           [1/Å]  (Q)  [0.8, 1, ..., 2.2, 2.4]
Data:
  L2 area                   float64            [meV]  (Q)  [0.998401, 1.02879, ..., 0.868326, 0.847639]  [0.00249147, 0.000493115, ..., 0.00339415, 0.00993478]
  L2 center                 float64            [meV]  (Q)  [0, 0, ..., 0, 0]  [0, 0, ..., 0, 0]
  L2 width                  float64            [meV]  (Q)  [0.0518672, 0.0748055, ..., 0.32826, 0.367889]  [1.09915e-05, 1.0944e-05, ..., 0.000613753, 0.00298264]

The dataset can be used like the one from `Analysis.parameters_to_dataset`. As in Tutorial 1, we fit the `L2` width to a Brownian diffusion model with a `ParameterAnalysis`.

In [13]:
brownian = edyn.BrownianTranslationalDiffusion(
    name='Brownian diffusion', lorentzian_name='L2', diffusion_coefficient=1e-9
)
binding = edyn.FitBinding(model=brownian, targets=['width'])

parameter_analysis = edyn.ParameterAnalysis(
    parameters=joint_analysis.parameters_to_dataset(), bindings=binding
)
parameter_analysis.fit()
parameter_analysis.plot()

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

The fitted diffusion coefficient. The data were generated with a width of $0.08\,\mathrm{meV\,Å^2}\cdot Q^2$, which corresponds to $D = 0.08\,\mathrm{meV\,Å^2}/\hbar \approx 1.2 \times 10^{-9}\,\mathrm{m^2/s}$.

In [14]:
brownian.get_global_variables()

[<Parameter 'diffusion_coefficient': 1.224e-09 m^2/s, bounds=[0.0:inf]>,
 <Parameter 'scale': 1.0000 meV, bounds=[0.0:inf]>]

## Sharing a diffusion model
Now that we know `L2` follows Brownian diffusion, we can fit the diffusion model directly to both data sets, as in Tutorial 1. A diffusion model object used in several sample models is shared too: its `Q` grid becomes the union of their `Q` values, each sample model uses the components at its own `Q` values, and the diffusion coefficient is fitted against all the data. The new sample models start from the original guesses for `L1`, `elastic` and `L3`. Create all the sample models before fitting: a new one can extend the grid, which regenerates the diffusion model's per-Q components and resets any per-Q parameters it has.

In [15]:
brownian = edyn.BrownianTranslationalDiffusion(
    name='Brownian diffusion', lorentzian_name='L2', diffusion_coefficient=1e-9
)
fine_model = edyn.SampleModel(components=L1, diffusion_models=brownian)
coarse_model = edyn.SampleModel(
    components=edyn.ComponentCollection(components=[elastic, L3]), diffusion_models=brownian
)

fine_analysis = edyn.Analysis(
    display_name='Fine resolution',
    experiment=fine_experiment,
    sample_model=fine_model,
    instrument_model=make_instrument_model(0.01),
)
coarse_analysis = edyn.Analysis(
    display_name='Coarse resolution',
    experiment=coarse_experiment,
    sample_model=coarse_model,
    instrument_model=make_instrument_model(0.1),
)

joint_analysis = edyn.JointAnalysis(analyses=[fine_analysis, coarse_analysis])
joint_analysis.fit()
fine_analysis.plot_data_and_model()

/home/runner/work/dynamics-lib/dynamics-lib/src/easydynamics/sample_model/component_collection.py:393: UserWarning: The Polynomial with unique_name Polynomial_46 has negative values, which may not be physically meaningful.
  gen = (component.evaluate(x, output=output) for component in self)


WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [16]:
brownian.get_global_variables()

[<Parameter 'diffusion_coefficient': 1.212e-09 ± 1.529e-11 m^2/s, bounds=[0.0:inf]>,
 <Parameter 'scale': 1.0241 ± 0.0083 meV, bounds=[0.0:inf]>]

`Q` values are considered equal within `Q_tolerance` (default 0.01 Å⁻¹). If the data sets have no `Q` values in common, nothing is shared and a warning is given; rebin the data onto a common `Q` grid to share components. Parameters that do not depend on `Q`, such as a diffusion coefficient, are not tied to the `Q` grid: a diffusion model object used in several sample models is shared across all of them, whatever their `Q` values. Its grid matches `Q` values exactly rather than within `Q_tolerance`, so two data sets at slightly different `Q` get two grid points.